In [148]:
from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import SystemMessage, HumanMessage
from langchain.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

# TODO (code)
- Evaluation -> different file
- logging of queries + results
- double checking actual tool use
- verifying model feasibility -> maybe switch to local model

# Config

# Init setup

In [149]:
GROQ_API_KEY = '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'

In [150]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b", #"qwen/qwen3.8-27b",
    api_key=GROQ_API_KEY,
    temperature=0
)

@dataclass
class Verification:
    verified: list[str] = field(default_factory=list) # claims that are verified to hold in the ontology
    violations: list[str] = field(default_factory=list) # claims where a contradiction or counter example exists
    not_verifiable: list[str] = field(default_factory=list) # claims that hold only due to the open world principle

@dataclass
class State:
    user_prompt: str
    tweets: Path
    summary_count: int = 0
    summary: str = ""
    verified: list[str] = field(default_factory=list)
    violations: list[str] = field(default_factory=list)
    not_verifiable: list[str] = field(default_factory=list)
    history: State = None
    # ^turns state into a linked list, bad for performance but will be fine for small sizes like here


# Nodes (Agentics and deterministic)

## Sumarization Agent

In [151]:
with open('System Prompt - Agent 1.md', 'r', encoding='utf-8') as f:
    summarization_system_prompt = f.read()

In [152]:
def summarize(state: State):
    # send prompt to LLM (no agent needed, this is a single linear call)
    response = llm.invoke([
        SystemMessage(summarization_system_prompt),
        HumanMessage(build_prompt(state.user_prompt, state.tweets))
    ])

    # make new state, storing historical state, incrementing the sumarycount and adding the LLM response
    return State(state.user_prompt, state.tweets, state.summary_count+1, response.content, history = state)

In [153]:
def build_prompt(user_prompt: str, tweets: Path):
    """"Assumes tweets is a complete filepath"""
    with open(tweets, 'r', encoding='utf-8') as f:
        return f"{user_prompt}\n{f.readlines()}"

## Verification Agent

In [154]:
with open('System Prompt - Agent 2.md', 'r', encoding='utf-8') as f:
    verification_system_prompt = f.read()

In [155]:
import random
@tool
def check_ontology(query: str) -> str:
    "checks if the query holds in the ontology, returns verified when entailed, violation when a counterexample or contradiction is found and not_verifyable when it holds only due to the open world assumption" 
    x = random.randint(0,2)
    if x == 0:
        return "verified"
    elif x ==1:
        return 'violation'
    else:
        return 'not_verifyable'

    # need to catch actual error and return them as text for the agent to retry failed queries
    

In [156]:
checker_agent = create_agent(
    llm,
    tools=[check_ontology],
    system_prompt=verification_system_prompt
)

formatter = llm.with_structured_output(Verification, method='json_schema') #TODO double check this is fine

c:\Users\Leonl\Documents\code projects\B1IA\B1IA\Lib\site-packages\pydantic\json_schema.py:2666: PydanticJsonSchemaWarning: Default value <factory> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


In [157]:
def verify(state: State):
    raw_result = checker_agent.invoke({'messages': [
            SystemMessage(verification_system_prompt),
            HumanMessage(state.summary)
        ]})
    result = formatter.invoke(raw_result['messages'])
    return result

## Best summary picker

In [158]:
def pick_best_summary(state: State):
    """picks the best summary based on number of violations"""
    def helper(current_state: State, best_state: State) -> State:
        if current_state.history is None:
            return best_state
        if len(current_state.violations) <= len(best_state.violations):
            return helper(current_state.history, current_state)
        else:
            return helper(current_state.history, best_state)
    return helper(state, state)

# recursive due to linked list, bit akward but it should work

# Routing Functions

In [159]:
def route(state: State, max_tries: int = 3): 
    "both routing diamonds in fig 1"
    if len(state.violations) == 0:
        return END
    elif state.summary_count < max_tries:
        return 'summarize'
    else:
        return 'pick_best_summary'

# Orchestration

In [160]:
graph = StateGraph(State)
graph.add_node('summarize', summarize)
graph.add_node('verify_summary', verify)
graph.add_node('pick_best_summary', pick_best_summary)
graph.add_edge(START, 'summarize')
graph.add_edge('summarize', 'verify_summary')
graph.add_conditional_edges('verify_summary', route)
graph.add_edge('pick_best_summary', END)

app = graph.compile()


# Final call to run

In [161]:
init_state = State(
    user_prompt='todo',
    tweets=Path('inputs/Netflix_clean.txt'))
out = app.invoke(init_state)

# Result

In [162]:
def print_summaries(state: State):
    print(f'Summary {state.summary_count}: \n{state.summary}')
    if state.history is None:
        return
    else:
        return print_summaries(state.history)
    
print_summaries(State(**out))

Summary 1: 
Several users describe spending leisure time alone watching Netflix in bed or on a couch while eating snacks or drinking, indicating it is a frequent way they unwind.  
Some tweets express frustration with technical issues such as Netflix not loading or lacking desired shows like “Super Troopers” or certain seasons, showing occasional dissatisfaction.  
Multiple tweets mention sharing or trading Netflix passwords, or feeling protective of their accounts, reflecting concerns about account access and cost.  
Positive remarks include excitement about specific content such as serial‑killer documentaries, “American Horror Story,” or childhood movies now available on Netflix, highlighting enjoyment of the platform’s library.  
A few tweets characterize Netflix as the primary or sole source of entertainment, with one user calling it “the only friend” and another noting they have stopped watching traditional TV because of commercials.
Summary 0: 

